# Training: PointNet++ + flow matching (~20 M parameters) on the unified datasets

Config `diffusion/configs/flow_pn2_20m.yaml`, data written by `dataset_tools/build_dataset.py` (e.g. GraspGen full + partial).

Before running:
1. Runtime → Change runtime type → **A100** (L4 / T4 work too, just slower).
2. Colab secrets (key icon on the left), notebook access on: `HF_TOKEN` (write access), `WANDB_API_KEY`, and `GH_TOKEN` if the repository is private.
3. Fill in the settings cell below.

In [ ]:
# ---- settings ----
REPO_URL   = 'https://github.com/JangoJangl/DIGNet.git'
BRANCH     = 'flow-matching-unified'
# Hugging Face dataset repos (or local folders, e.g. copied from Google Drive)
DATA_PATHS = ['Thetatec/graspgen-unified-full', 'Thetatec/graspgen-unified-partial']
HUB_REPO   = 'Thetatec/dignet-flow-pn2-20m'   # model repo for checkpoints + training state, None to disable
WANDB_ENTITY  = None                           # None: your default wandb entity
WANDB_PROJECT = 'dignet'
RUN_DIR    = '/content/drive/MyDrive/dignet_runs'   # hydra run folders; on Drive they survive a disconnect
MAX_STEPS  = 300_000
EXTRA_OVERRIDES = []                           # e.g. ['num_objects=32', 'agents.optimization.lr=3e-4']
# resume an interrupted run: '<HUB_REPO>:runs/<wandb run name>/last/training_state.pth' and its wandb run id
RESUME, WANDB_RUN_ID = None, None

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata, drive
for key in ('HF_TOKEN', 'WANDB_API_KEY', 'GH_TOKEN'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'secret {key} not set')
if RUN_DIR.startswith('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
url = REPO_URL.replace('https://', f"https://{os.environ['GH_TOKEN']}@") if os.environ.get('GH_TOKEN') else REPO_URL
!rm -rf /content/DIGNet && git clone -q --depth 1 -b {BRANCH} {url} /content/DIGNet
# Colab's torch is used as is; pytorch3d and alr_sim are optional (alr_sim only adds the gripper collision metric)
!pip install -q hydra-core omegaconf wandb einops trimesh open3d rtree h5py scikit-learn theseus-ai datasets huggingface_hub matplotlib

In [ ]:
import sys
sys.path[:0] = ['/content/DIGNet', '/content/DIGNet/diffusion']
import theseus, open3d, datasets, hydra  # noqa: F401  fails here if an install went wrong
from diffusion.environments.diffusion_grasp.unified_dataset import load_unified
for p in DATA_PATHS:  # downloads the HF datasets once into the cache and shows the splits
    print(p, {s: len(load_unified(p, s) or []) for s in ('train', 'validation', 'test')})

## Train

`checkpoint_every_n_steps` saves model, optimizer, EMA and step to `<run>/last/training_state.pth` and pushes it to `HUB_REPO`, so a Colab disconnect costs at most that many steps. Prediction plots appear in wandb at 25/50/75/100 %.

In [ ]:
overrides = [
    f"unified.paths=[{','.join(DATA_PATHS)}]",
    'device=cuda',
    f'max_train_steps={MAX_STEPS}',
    'checkpoint_every_n_steps=2000',
    f'log_dir={RUN_DIR}',
    f'wandb.project={WANDB_PROJECT}',
    f"wandb.entity={WANDB_ENTITY or 'null'}",
    f"hub.repo_id={HUB_REPO or 'null'}",
] + EXTRA_OVERRIDES
if RESUME:
    overrides += [f'resume={RESUME}', f'wandb.run_id={WANDB_RUN_ID}']
args = ' '.join(f'"{o}"' for o in overrides)
%cd /content/DIGNet/diffusion
!PYTHONPATH=/content/DIGNet:/content/DIGNet/diffusion python training.py --config-name flow_pn2_20m {args}

## Resume after a disconnect

1. Look up the wandb run name and id (run overview page).
2. Set `RESUME = '<HUB_REPO>:runs/<run name>/last/training_state.pth'` and `WANDB_RUN_ID = '<id>'` in the settings cell.
3. Run all cells again. Training continues at the saved step with optimizer and EMA state, and logs into the same wandb run.